In [ ]:
import os
import pandas as pd
import numpy as np

from mp_api.client import MPRester
from matminer.featurizers.structure import (
    SiteStatsFingerprint,
    BagofBonds
)
from matminer.featurizers.structure.matrix import SineCoulombMatrix


# ============================================================
# 1. LOAD YOUR CSV FILE
# ============================================================

input_file = "perovskites_data.csv"

df = pd.read_csv(input_file)

print("✅ Dataset loaded successfully")
print("Number of rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. CHECK REQUIRED COLUMN
# ============================================================

if "Material_ID" not in df.columns:
    raise KeyError(
        "❌ Material_ID column not found.\n"
        "Available columns:\n"
        + str(df.columns.tolist())
    )


# ============================================================
# 3. CLEAN MATERIAL IDs
# ============================================================

df["Material_ID"] = df["Material_ID"].astype(str).str.strip()

material_ids = df["Material_ID"].dropna().unique().tolist()

print("\nNumber of unique Materials Project IDs:", len(material_ids))


# ============================================================
# 4. GET CRYSTAL STRUCTURES FROM MATERIALS PROJECT
# ============================================================

# Paste your Materials Project API key directly below
API_KEY = "yG9MO3EiaegBkJOVJauCFubkj8QJgeGr"


print("\nDownloading crystal structures from Materials Project...")

structures = {}

with MPRester(API_KEY) as mpr:

    results = mpr.materials.summary.search(
        material_ids=material_ids,
        fields=["material_id", "structure"]
    )

    for entry in results:

        if entry.structure is not None:
            structures[str(entry.material_id)] = entry.structure


print(
    f"✅ Retrieved {len(structures)} crystal structures "
    f"out of {len(material_ids)} material IDs."
)


# ============================================================
# 5. ADD STRUCTURES TO DATAFRAME
# ============================================================

df["structure"] = df["Material_ID"].map(structures)


# Check missing structures
missing_structures = df["structure"].isna().sum()

print("Missing structures:", missing_structures)


# Remove rows where structure could not be obtained
df = df.dropna(subset=["structure"]).reset_index(drop=True)

print(
    f"Remaining structures available for featurization: {len(df)}"
)


# ============================================================
# 6. SITE STATS FINGERPRINT
# ============================================================

print("\n==========================================")
print("Generating SiteStatsFingerprint features")
print("==========================================")

ssf = SiteStatsFingerprint.from_preset(
    "CoordinationNumber_ward-prb-2017"
)

# Parallelize across available CPU cores for speed (logic unchanged)
ssf.set_n_jobs(max(1, (os.cpu_count() or 2) - 1))

df_ssf = ssf.featurize_dataframe(
    df[["structure"]].copy(),
    col_id="structure",
    ignore_errors=True
)

print(
    "✅ Generated",
    df_ssf.shape[1] - 1,
    "SiteStatsFingerprint features."
)


# ============================================================
# 7. BAG OF BONDS
# ============================================================

print("\n==========================================")
print("Generating Bag of Bonds features")
print("==========================================")

# FIX: BagofBonds() defaults to the molecular (non-periodic) CoulombMatrix,
# which is invalid for periodic crystal Structures and is what caused the
# featurizer to hang / blow up while building its bond vocabulary across
# 4,719 structures. SineCoulombMatrix is the periodic-aware equivalent and
# is the correct choice for crystalline materials.
bob = BagofBonds(coulomb_matrix=SineCoulombMatrix())
bob.set_n_jobs(max(1, (os.cpu_count() or 2) - 1))

# BoB must first learn the bond types present
# in the complete structure dataset
bob.fit(df["structure"].tolist())

df_bob = bob.featurize_dataframe(
    df[["structure"]].copy(),
    col_id="structure",
    ignore_errors=True
)

print(
    "✅ Generated",
    df_bob.shape[1] - 1,
    "Bag of Bonds features."
)


# ============================================================
# 8. REMOVE STRUCTURE COLUMN
# ============================================================

ssf_features = df_ssf.drop(
    columns=["structure"],
    errors="ignore"
)

bob_features = df_bob.drop(
    columns=["structure"],
    errors="ignore"
)


# ============================================================
# 9. COMBINE STRUCTURAL FEATURES
# ============================================================

df_structural = pd.concat(
    [
        ssf_features,
        bob_features
    ],
    axis=1
)


# ============================================================
# 10. REMOVE COMPLETELY EMPTY FEATURES
# ============================================================

df_structural = df_structural.dropna(
    axis=1,
    how="all"
)


# ============================================================
# 11. SAVE STRUCTURAL FEATURES
# ============================================================

output_file = "structural_features_s22.csv"

df_structural.to_csv(
    output_file,
    index=False
)


# ============================================================
# 12. FINAL INFORMATION
# ============================================================

print("\n==========================================")
print("✅ STRUCTURAL FEATURE GENERATION COMPLETE")
print("==========================================")

print("Output file:", output_file)
print("Rows:", df_structural.shape[0])
print("Features:", df_structural.shape[1])

✅ Dataset loaded successfully
Number of rows: 4719

Columns:
['Material_ID', 'Formula', 'Bandgap', 'Formation_Energy', 'Volume']

Number of unique Materials Project IDs: 4719



Retrieving 4719 material_ids values in 47 batches:   0%|          | 0/47 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

...

Retrieving SummaryDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/19 [00:00<?, ?it/s]

✅ Retrieved 4719 crystal structures out of 4719 material IDs.
Missing structures: 0
Remaining structures available for featurization: 4719

Generating SiteStatsFingerprint features


SiteStatsFingerprint:   0%|          | 0/4719 [00:00<?, ?it/s]